# Ejemplo de Estrategia Swap Spread a 5Y

In [1]:
from collections import namedtuple
import pandas as pd

import qcfinancial as qcf

import aux_functions as aux
import qcf_wrappers as qcw

In [2]:
%load_ext autoreload
%autoreload 2

## Carga de Curvas de Mercado

In [3]:
df_cicpclp = pd.read_excel("input/20250926_cicpclp.xlsx")
df_cclpcolusd = pd.read_csv("input/20250926_cclpcolusd.csv")
df_cclfcolusd = pd.read_excel("input/20250926_cclfcolusd.xlsx")

Se construyen los objetos `ZeroCouponCurve` de `qcfinancial`.

In [4]:
qcf_cicpclp = aux.build_zero_coupon_curve(
    df_cicpclp.maturity.values,
    df_cicpclp.rate.values,
    cyf := qcw.YearFraction.ACT365,
    cwf := qcw.WealthFactor.CON,
)

In [5]:
qcf_cclpcolusd = aux.build_zero_coupon_curve(
    df_cclpcolusd.maturity.values,
    df_cclpcolusd.rate.values,
    cyf,
    cwf,
)

In [6]:
qcf_cclfcolusd = aux.build_zero_coupon_curve(
    df_cclfcolusd.maturity.values,
    df_cclfcolusd.rate.values,
    cyf,
    cwf,
)

In [7]:
notional_uf = 12_500_000.0

## BTU a 5Y

In [8]:
scl = aux.get_business_calendar("CL", range(2018, 2031))

In [9]:
params_bono = dict(
    rec_pay = qcf.RecPay.RECEIVE,
    start_date = qcf.QCDate(1, 3, 2018),
    end_date = qcf.QCDate(1, 9, 2030),
    bus_adj_rule = qcf.BusyAdjRules.NO,
    settlement_periodicity = qcf.Tenor('6M'),
    settlement_stub_period = qcf.StubPeriod.NO,
    settlement_calendar = scl,
    settlement_lag = 0,
    initial_notional = notional_uf,
    amort_is_cashflow = True,
    interest_rate = qcf.QCInterestRate(
        0.019,
        qcf.QC30360(),
        qcf.QCLinearWf(),
    ),
    notional_currency = qcf.QCCLF(),
    is_bond = True,
    sett_lag_behaviour = qcf.SettLagBehaviour.DONT_MOVE,
)

# Se da de alta el objeto
pata_bono = qcf.LegFactory.build_bullet_fixed_rate_leg(**params_bono)

In [10]:
tera = qcf.QCInterestRate(.019071, qcf.QCAct365(), qcf.QCCompoundWf())
bono_chileno = qcf.ChileanFixedRateBond(pata_bono, tera)

In [11]:
aux.leg_as_dataframe(pata_bono)

,fecha_inicial,fecha_final,fecha_pago,nocional,amortizacion,interes,amort_es_flujo,flujo,moneda_nocional,valor_tasa,tipo_tasa
0,2018-03-01,2018-09-01,2018-09-01,12500000.0,0.0,118750.0,True,118750.0,CLF,0.019,Lin30360
1,2018-09-01,2019-03-01,2019-03-01,12500000.0,0.0,118750.0,True,118750.0,CLF,0.019,Lin30360
2,2019-03-01,2019-09-01,2019-09-01,12500000.0,0.0,118750.0,True,118750.0,CLF,0.019,Lin30360
3,2019-09-01,2020-03-01,2020-03-01,12500000.0,0.0,118750.0,True,118750.0,CLF,0.019,Lin30360
4,2020-03-01,2020-09-01,2020-09-01,12500000.0,0.0,118750.0,True,118750.0,CLF,0.019,Lin30360
5,2020-09-01,2021-03-01,2021-03-01,12500000.0,0.0,118750.0,True,118750.0,CLF,0.019,Lin30360
6,2021-03-01,2021-09-01,2021-09-01,12500000.0,0.0,118750.0,True,118750.0,CLF,0.019,Lin30360
7,2021-09-01,2022-03-01,2022-03-01,12500000.0,0.0,118750.0,True,118750.0,CLF,0.019,Lin30360
8,2022-03-01,2022-09-01,2022-09-01,12500000.0,0.0,118750.0,True,118750.0,CLF,0.019,Lin30360
9,2022-09-01,2023-03-01,2023-03-01,12500000.0,0.0,118750.0,True,118750.0,CLF,0.019,Lin30360


## Swap ICP a 5Y

Hay que dar de alta la pata fija (CLF) y la pata ICPCLP.

In [12]:
ny = aux.get_business_calendar("US", range(2025, 2030))

In [13]:
params_pata_fija_swap = dict(
    rec_pay = qcf.RecPay.PAY,
    start_date = (start_date := qcf.QCDate(30, 9, 2025)),
    end_date = (end_date := qcf.QCDate(1, 9, 2030)),
    bus_adj_rule = (bus_adj_rule := qcf.BusyAdjRules.MODFOLLOW),
    settlement_periodicity = (settlement_periodicity := qcf.Tenor('6M')),
    settlement_stub_period = (settlement_stub_period := qcf.StubPeriod.SHORTFRONT),
    settlement_calendar = (settlement_calendar := (scl + ny)),
    settlement_lag = (settlement_lag := 0),
    initial_notional = notional_uf,
    amort_is_cashflow = (amort_is_cashflow := True),
    interest_rate = qcf.QCInterestRate(
        # 0.016450,
        0.016385,
        qcf.QCAct360(),
        qcf.QCLinearWf(),
    ),
    notional_currency = qcf.QCCLF(),
    is_bond = False,
    sett_lag_behaviour = (sett_lag_behaviour := qcf.SettLagBehaviour.DONT_MOVE),
)

pata_fija_swap = qcf.LegFactory.build_bullet_fixed_rate_leg(**params_pata_fija_swap)

In [14]:
aux.leg_as_dataframe(pata_fija_swap)

,fecha_inicial,fecha_final,fecha_pago,nocional,amortizacion,interes,amort_es_flujo,flujo,moneda_nocional,valor_tasa,tipo_tasa
0,2025-09-30,2026-03-02,2026-03-02,-12500000.0,0.0,-87045.312500,True,-8.704531e+04,CLF,0.016385,LinAct360
1,2026-03-02,2026-09-01,2026-09-01,-12500000.0,0.0,-104113.020833,True,-1.041130e+05,CLF,0.016385,LinAct360
2,2026-09-01,2027-03-01,2027-03-01,-12500000.0,0.0,-102975.173611,True,-1.029752e+05,CLF,0.016385,LinAct360
3,2027-03-01,2027-09-01,2027-09-01,-12500000.0,0.0,-104681.944444,True,-1.046819e+05,CLF,0.016385,LinAct360
4,2027-09-01,2028-03-01,2028-03-01,-12500000.0,0.0,-103544.097222,True,-1.035441e+05,CLF,0.016385,LinAct360
5,2028-03-01,2028-09-01,2028-09-01,-12500000.0,0.0,-104681.944444,True,-1.046819e+05,CLF,0.016385,LinAct360
6,2028-09-01,2029-03-01,2029-03-01,-12500000.0,0.0,-102975.173611,True,-1.029752e+05,CLF,0.016385,LinAct360
7,2029-03-01,2029-09-04,2029-09-04,-12500000.0,0.0,-106388.715278,True,-1.063887e+05,CLF,0.016385,LinAct360
8,2029-09-04,2030-03-01,2030-03-01,-12500000.0,0.0,-101268.402778,True,-1.012684e+05,CLF,0.016385,LinAct360
9,2030-03-01,2030-09-02,2030-09-02,-12500000.0,-12500000.0,-105250.868056,True,-1.260525e+07,CLF,0.016385,LinAct360


In [15]:
params_pata_icpclp = dict(
    rec_pay=qcf.RecPay.RECEIVE,
    start_date=start_date,
    end_date=end_date,
    bus_adj_rule=bus_adj_rule,
    fix_adj_rule=qcf.BusyAdjRules.PREVIOUS,
    settlement_periodicity=settlement_periodicity,
    settlement_stub_period=settlement_stub_period,
    settlement_calendar=settlement_calendar,
    fixing_calendar=scl,
    settlement_lag=settlement_lag,
    initial_notional= (notional_clp:=notional_uf * 39_485.65),
    amort_is_cashflow=amort_is_cashflow,
    spread=0.0,
    gearing=1.0,
    interest_rate = qcf.QCInterestRate(
        0.0,
        qcf.QCAct360(),
        qcf.QCLinearWf(),
    ),
    index_name="ICPCLP",
    eq_rate_decimal_places=4,
    notional_currency=qcf.QCCLP(),
    dates_for_eq_rate=qcf.DatesForEquivalentRate.ACCRUAL,
    sett_lag_behaviour=sett_lag_behaviour,
)

pata_icpclp_swap = qcf.LegFactory.build_bullet_overnight_index_leg(**params_pata_icpclp)

## Valorización al 2025-09-26

In [16]:
tir_mercado = {
    (fecha_hoy := qcf.QCDate("2025-09-26")): qcf.QCInterestRate(.022319, qcf.QCAct365(), qcf.QCCompoundWf()),
}

tir_mercado_up = {
    qcf.QCDate("2025-09-26"): qcf.QCInterestRate(.022419, qcf.QCAct365(), qcf.QCCompoundWf()),
}

In [17]:
ufs = {
    fecha_hoy: 39_485.65,
    start_date: 39_485.65,
}

icp = {
    fecha_hoy: 25_713.89,
    start_date: 25_724.07,
}


In [18]:
pv = qcf.PresentValue()
fwd = qcf.ForwardRates()

Valor presente y sensibilidad de toda la estructura.

In [19]:
valor_presente_base = aux.vp_swap_spread(
    fecha_hoy,
    # Bono
    pata_bono,
    tir_mercado[fecha_hoy],
    tir_mercado[fecha_hoy],
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp,
    qcf_cclpcolusd,
    ufs[fecha_hoy],
    icp[fecha_hoy],
    icp[fecha_hoy],
    pv,
    fwd,
)

493570625000.0


In [55]:
print("###### Bono ######")
print(f"Valor Presente: {valor_presente_base.pv_bono:,.0f}")
print(f"Sensibilidad: {valor_presente_base.sens_bono:,.0f}")
print(f"Aj. Valor Mercado: {valor_presente_base.ajuste_valor_mercado_bono:,.0f}")

print("\n###### Pata Fija Swap ######")
print(f"Valor Presente: {valor_presente_base.pv_pata_fija_swap:,.0f}")
print(f"K + Interés Devengado: {valor_presente_base.accrued_value_fija_swap:,.0f}")
print(f"Sensibilidad: {valor_presente_base.sens_pata_fija_swap:,.0f}")

print("\n###### Pata ICPCLP Swap ######")
print(f"Valor Presente: {valor_presente_base.pv_pata_icpclp:,.0f}")
print(f"K + Interés Devengado: {valor_presente_base.accrued_value_icpclp_swap:,.0f}")
print(f"Sensibilidad Proyección: {valor_presente_base.sens_proyeccion_pata_icpclp_swap:,.0f}")
print(f"Sensibilidad Descuento: {valor_presente_base.sens_descuento_pata_icpclp_swap:,.0f}")

print("\n###### Todo el Swap ######")
print(f"Valor Presente: {valor_presente_base.pv_pata_fija_swap + valor_presente_base.pv_pata_icpclp:,.0f}")
print(f"Sensibilidad: {valor_presente_base.sens_pata_fija_swap +
                       valor_presente_base.sens_proyeccion_pata_icpclp_swap +
                       valor_presente_base.sens_descuento_pata_icpclp_swap:,.0f}")

print(f"\nMark to Market Total: {valor_presente_base.valor_presente_total:,.0f}")
aj_total = (
        valor_presente_base.ajuste_valor_mercado_bono +
        valor_presente_base.pv_pata_fija_swap - valor_presente_base.accrued_value_fija_swap +
        valor_presente_base.pv_pata_icpclp - valor_presente_base.accrued_value_icpclp_swap
)
print(f"Ajuste a Valor de Mercado Total: {aj_total:,.0f}")

###### Bono ######
Valor Presente: 486,853,780,651
Sensibilidad: -225,038,576
Aj. Valor Mercado: 0

###### Pata Fija Swap ######
Valor Presente: -499,640,829,429
K + Interés Devengado: -493,570,625,000
Sensibilidad: 237,886,045

###### Pata ICPCLP Swap ######
Valor Presente: 499,641,877,208
K + Interés Devengado: 493,570,625,000
Sensibilidad Proyección: 219,516,487
Sensibilidad Descuento: -223,006,383

###### Todo el Swap ######
Valor Presente: 1,047,779
Sensibilidad: 234,396,149

Mark to Market Total: 486,854,828,430
Ajuste a Valor de Mercado Total: 1,047,779


## Cálculo de Swap Spread

In [21]:
pv_bono_swap = pv.pv(fecha_hoy, pata_bono, qcf_cclfcolusd) * ufs[fecha_hoy]
print(f"PV Bono con Swap: {pv_bono_swap:,.0f}")

PV Bono con Swap: 505,982,394,848


In [22]:
tir_swap = aux.encuentra_tir_swap(fecha_hoy, pata_bono, 0.019, pv_bono_swap / ufs[fecha_hoy], pv)
print(f"tir_swap: {tir_swap:,.4%}")
print(f"Swap Spread: {(swap_spread := tir_mercado[fecha_hoy].get_value() - tir_swap):,.4%}")

tir_swap: 1.4020%
Swap Spread: 0.8299%


In [23]:
bp = 1
swap_spread_up = swap_spread + bp / 10_000.0
print(f"Swap Spread + {bp} pb: {swap_spread_up:.4%}")
tir_mercado_up = qcf.QCInterestRate(tir_swap + swap_spread_up, qcf.QCAct365(), qcf.QCCompoundWf())

Swap Spread + 1 pb: 0.8399%


In [24]:
valor_presente_up = aux.vp_swap_spread(
    fecha_hoy,
    # Bono
    pata_bono,
    tir_mercado_up,
    tir_mercado[fecha_hoy],
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp,
    qcf_cclpcolusd,
    ufs[fecha_hoy],
    icp[fecha_hoy],
    icp[fecha_hoy],
    pv,
    fwd,
)

493570625000.0


In [25]:
print(f"PV Bono: {valor_presente_up.pv_bono:,.0f}")
print(f"Sens Bono: {valor_presente_up.sens_bono:,.0f}")
print(f"\nPV Pata Fija Swap: {valor_presente_up.pv_pata_fija_swap:,.0f}")
print(f"Sens Pata Fija Swap: {valor_presente_up.sens_pata_fija_swap:,.0f}")
print(f"\nPV Pata ICPCLP: {valor_presente_up.pv_pata_icpclp:,.0f}")
print(f"Sens Proyección Pata ICPCLP: {valor_presente_up.sens_proyeccion_pata_icpclp_swap:,.0f}")
print(f"Sens Descuento ICPCLP: {valor_presente_up.sens_descuento_pata_icpclp_swap:,.0f}")
print(f"\nMark to Market Total: {valor_presente_up.valor_presente_total:,.0f}")
print(f"Delta: {valor_presente_up.valor_presente_total - valor_presente_base.valor_presente_total:,.0f}")

PV Bono: 486,628,806,530
Sens Bono: -224,909,681

PV Pata Fija Swap: -499,640,829,429
Sens Pata Fija Swap: 237,886,045

PV Pata ICPCLP: 499,641,877,208
Sens Proyección Pata ICPCLP: 219,516,487
Sens Descuento ICPCLP: -223,006,383

Mark to Market Total: 486,629,854,309
Delta: -224,974,122


In [26]:
swap_spread_down = swap_spread - bp / 10_000.0
print(f"Swap Spread - {bp} pb: {swap_spread_down:.4%}")
tir_mercado_down = qcf.QCInterestRate(tir_swap + swap_spread_down, qcf.QCAct365(), qcf.QCCompoundWf())

Swap Spread - 1 pb: 0.8199%


In [27]:
valor_presente_down = aux.vp_swap_spread(
    fecha_hoy,
    # Bono
    pata_bono,
    tir_mercado_down,
    tir_mercado[fecha_hoy],
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp,
    qcf_cclpcolusd,
    ufs[fecha_hoy],
    icp[fecha_hoy],
    icp[fecha_hoy],
    pv,
    fwd,
)

493570625000.0


In [28]:
print(f"PV Bono: {valor_presente_down.pv_bono:,.0f}")
print(f"Sens Bono: {valor_presente_down.sens_bono:,.0f}")
print(f"\nPV Pata Fija Swap: {valor_presente_down.pv_pata_fija_swap:,.0f}")
print(f"Sens Pata Fija Swap: {valor_presente_down.sens_pata_fija_swap:,.0f}")
print(f"\nPV Pata ICPCLP: {valor_presente_down.pv_pata_icpclp:,.0f}")
print(f"Sens Proyección Pata ICPCLP: {valor_presente_down.sens_proyeccion_pata_icpclp_swap:,.0f}")
print(f"Sens Descuento ICPCLP: {valor_presente_down.sens_descuento_pata_icpclp_swap:,.0f}")
print(f"\nMark to Market Total: {valor_presente_down.valor_presente_total:,.0f}")
print(f"Delta: {valor_presente_down.valor_presente_total - valor_presente_base.valor_presente_total:,.0f}")

PV Bono: 487,078,883,711
Sens Bono: -225,167,559

PV Pata Fija Swap: -499,640,829,429
Sens Pata Fija Swap: 237,886,045

PV Pata ICPCLP: 499,641,877,208
Sens Proyección Pata ICPCLP: 219,516,487
Sens Descuento ICPCLP: -223,006,383

Mark to Market Total: 487,079,931,491
Delta: 225,103,060


## Escenarios de Curvas

In [29]:
qcf_cicpclp_1 = aux.build_zero_coupon_curve(
    df_cicpclp.maturity.values,
    df_cicpclp.rate_1.values,
    cyf := qcw.YearFraction.ACT365,
    cwf := qcw.WealthFactor.CON,
)

In [30]:
valor_presente_esc_1 = aux.vp_swap_spread(
    fecha_hoy,
    # Bono
    pata_bono,
    tir_mercado[fecha_hoy],
    tir_mercado[fecha_hoy],
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp_1,
    qcf_cclpcolusd,
    ufs[fecha_hoy],
    icp[fecha_hoy],
    icp[fecha_hoy],
    pv,
    fwd,
)

493570625000.0


In [31]:
print(f"PV Bono: {valor_presente_esc_1.pv_bono:,.0f}")
print(f"Sens Bono: {valor_presente_esc_1.sens_bono:,.0f}")
print(f"\nPV Pata Fija Swap: {valor_presente_esc_1.pv_pata_fija_swap:,.0f}")
print(f"Sens Pata Fija Swap: {valor_presente_esc_1.sens_pata_fija_swap:,.0f}")
print(f"\nPV Pata ICPCLP: {valor_presente_esc_1.pv_pata_icpclp:,.0f}")
print(f"Sens Proyección Pata ICPCLP: {valor_presente_esc_1.sens_proyeccion_pata_icpclp_swap:,.0f}")
print(f"Sens Descuento ICPCLP: {valor_presente_esc_1.sens_descuento_pata_icpclp_swap:,.0f}")
print(f"\nMark to Market Total: {valor_presente_esc_1.valor_presente_total:,.0f}")
print(f"Delta: {valor_presente_esc_1.valor_presente_total - valor_presente_base.valor_presente_total:,.0f}")

PV Bono: 486,853,780,651
Sens Bono: -225,038,576

PV Pata Fija Swap: -499,640,829,429
Sens Pata Fija Swap: 237,886,045

PV Pata ICPCLP: 521,854,767,763
Sens Proyección Pata ICPCLP: 214,905,967
Sens Descuento ICPCLP: -228,848,879

Mark to Market Total: 509,067,718,985
Delta: 22,212,890,554


## Escenarios de Curva y Spread Después de 1 Mes

In [32]:
fecha_1_mes = qcf.QCDate("2025-10-27")
uf_1_mes = 39_577.28
icp_1_mes = icp[fecha_hoy] * (1 + .0475 * 32 / 360)
tir_compra = tir_mercado[fecha_hoy].get_value()

### Sube Curva UF mantiene Spread

In [62]:
bp = 100
tir_mercado = aux.make_tir(tir_swap + bp / 10_000.0 + swap_spread)

In [63]:
qcf_cclfcolusd_1 = aux.build_zero_coupon_curve(
    df_cclfcolusd.maturity.values,
    df_cclfcolusd.rate_1.values,
    cyf := qcw.YearFraction.ACT365,
    cwf := qcw.WealthFactor.CON,
)

In [64]:
sube_curva_igual_spread = aux.vp_swap_spread(
    fecha_1_mes,
    # Bono
    pata_bono,
    tir_mercado,
    aux.make_tir(tir_compra),
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd_1,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp,
    qcf_cclpcolusd,
    uf_1_mes,
    icp[start_date],
    icp_1_mes,
    pv,
    fwd,
)

493570625000.0


In [65]:
fff = "{:,.0f}"
format_dict = {
    "pv_bono": fff,
}

In [66]:
ForDF = namedtuple('ForDF', [
    'pv_bono',
    'pv_compra_bono',
    'ajuste_valor_mercado_bono',
    'pv_pata_fija_swap',
    'accrued_value_fija_swap',
    'ajuste_valor_mercado_fija_swap',
    'pv_pata_icpclp',
    'accrued_value_icpclp_swap',
    'ajuste_valor_mercado_icpclp_swap',
    'valor_presente_total',
])

In [67]:
for_df = ForDF(
    pv_bono=sube_curva_igual_spread.pv_bono,
    pv_compra_bono=sube_curva_igual_spread.pv_compra_bono,
    ajuste_valor_mercado_bono=sube_curva_igual_spread.ajuste_valor_mercado_bono,
    pv_pata_fija_swap=(pv_mkt_fija_swap:=sube_curva_igual_spread.pv_pata_fija_swap),
    accrued_value_fija_swap=(accrued_fija_swap:=sube_curva_igual_spread.accrued_value_fija_swap),
    ajuste_valor_mercado_fija_swap=pv_mkt_fija_swap - accrued_fija_swap,
    pv_pata_icpclp=(pv_mkt_icpclp_swap:=sube_curva_igual_spread.pv_pata_icpclp),
    accrued_value_icpclp_swap=(accrued_icpclp_swap:=sube_curva_igual_spread.accrued_value_icpclp_swap),
    ajuste_valor_mercado_icpclp_swap=pv_mkt_icpclp_swap - accrued_icpclp_swap,
    valor_presente_total=sube_curva_igual_spread.valor_presente_total,
)

In [68]:
pd.DataFrame([for_df]).style.format(fff)

,pv_bono,pv_compra_bono,ajuste_valor_mercado_bono,pv_pata_fija_swap,accrued_value_fija_swap,ajuste_valor_mercado_fija_swap,pv_pata_icpclp,accrued_value_icpclp_swap,ajuste_valor_mercado_icpclp_swap,valor_presente_total
0,"467,320,190,444","488,899,270,026","-21,579,079,582","-478,724,691,286","-495,323,944,124","16,599,252,838","501,749,505,243","495,458,532,641","6,290,972,603","490,345,004,401"


In [69]:
devengo_bono = sube_curva_igual_spread.pv_compra_bono - valor_presente_base.pv_compra_bono
print(f"Devengo Bono: {devengo_bono:,.0f}")

Devengo Bono: 2,045,489,375


In [70]:
devengo_fija_swap = sube_curva_igual_spread.accrued_value_fija_swap - valor_presente_base.accrued_value_fija_swap
devengo_icpclp_swap = sube_curva_igual_spread.accrued_value_icpclp_swap - valor_presente_base.accrued_value_icpclp_swap
print(f"Devengo Swap: {devengo_fija_swap + devengo_icpclp_swap:,.0f}")

Devengo Swap: 134,588,516


In [71]:
ajuste_bono = sube_curva_igual_spread.ajuste_valor_mercado_bono
print(f"Ajuste a Valor de Mercado Bono: {ajuste_bono:,.0f}")

Ajuste a Valor de Mercado Bono: -21,579,079,582


In [72]:
ajuste_fija_swap = sube_curva_igual_spread.pv_pata_fija_swap - sube_curva_igual_spread.accrued_value_fija_swap
print(f"Ajuste a Valor de Mercado Fija Swap: {ajuste_fija_swap:,.0f}")

Ajuste a Valor de Mercado Fija Swap: 16,599,252,838


In [73]:
ajuste_icpclp_swap = sube_curva_igual_spread.pv_pata_icpclp - sube_curva_igual_spread.accrued_value_icpclp_swap
print(f"Ajuste a Valor de Mercado ICPCLP Swap: {ajuste_icpclp_swap:,.0f}")

Ajuste a Valor de Mercado ICPCLP Swap: 6,290,972,603


In [75]:
print(f"Ajuste a Valor de Mercado Swap: {ajuste_fija_swap + ajuste_icpclp_swap:,.0f}")

Ajuste a Valor de Mercado Swap: 22,890,225,441


### Mantiene Curva UF Sube Spread

In [87]:
bp = 100
tir_mercado = aux.make_tir(tir_swap + bp / 10_000.0 + swap_spread)

In [99]:
igual_curva_sube_spread = aux.vp_swap_spread(
    fecha_1_mes,
    # Bono
    pata_bono,
    tir_mercado,
    aux.make_tir(tir_compra),
    # Pata Fija Swap
    pata_fija_swap,
    qcf_cclfcolusd,
    # Pata ICPCLP Swap
    pata_icpclp_swap,
    qcf_cicpclp,
    qcf_cclpcolusd,
    uf_1_mes,
    icp[start_date],
    icp_1_mes,
    pv,
    fwd,
)

493570625000.0


In [100]:
for_df = ForDF(
    pv_bono=igual_curva_sube_spread.pv_bono,
    pv_compra_bono=igual_curva_sube_spread.pv_compra_bono,
    ajuste_valor_mercado_bono=igual_curva_sube_spread.ajuste_valor_mercado_bono,
    pv_pata_fija_swap=(pv_mkt_fija_swap:=igual_curva_sube_spread.pv_pata_fija_swap),
    accrued_value_fija_swap=(accrued_fija_swap:=igual_curva_sube_spread.accrued_value_fija_swap),
    ajuste_valor_mercado_fija_swap=pv_mkt_fija_swap - accrued_fija_swap,
    pv_pata_icpclp=(pv_mkt_icpclp_swap:=igual_curva_sube_spread.pv_pata_icpclp),
    accrued_value_icpclp_swap=(accrued_icpclp_swap:=igual_curva_sube_spread.accrued_value_icpclp_swap),
    ajuste_valor_mercado_icpclp_swap=pv_mkt_icpclp_swap - accrued_icpclp_swap,
    valor_presente_total=igual_curva_sube_spread.valor_presente_total,
)

In [101]:
pd.DataFrame([for_df]).style.format(fff)

,pv_bono,pv_compra_bono,ajuste_valor_mercado_bono,pv_pata_fija_swap,accrued_value_fija_swap,ajuste_valor_mercado_fija_swap,pv_pata_icpclp,accrued_value_icpclp_swap,ajuste_valor_mercado_icpclp_swap,valor_presente_total
0,"467,320,190,444","488,899,270,026","-21,579,079,582","-501,630,063,434","-495,323,944,124","-6,306,119,309","501,749,505,243","495,458,532,641","6,290,972,603","467,439,632,254"


In [103]:
devengo_bono = igual_curva_sube_spread.pv_compra_bono - valor_presente_base.pv_compra_bono
print(f"Devengo Bono: {devengo_bono:,.0f}")

Devengo Bono: 2,045,489,375


In [104]:
devengo_fija_swap = igual_curva_sube_spread.accrued_value_fija_swap - valor_presente_base.accrued_value_fija_swap
devengo_icpclp_swap = igual_curva_sube_spread.accrued_value_icpclp_swap - valor_presente_base.accrued_value_icpclp_swap
print(f"Devengo Swap: {devengo_fija_swap + devengo_icpclp_swap:,.0f}")

Devengo Swap: 134,588,516


In [105]:
ajuste_bono = igual_curva_sube_spread.ajuste_valor_mercado_bono
print(f"Ajuste a Valor de Mercado Bono: {ajuste_bono:,.0f}")

Ajuste a Valor de Mercado Bono: -21,579,079,582


In [106]:
ajuste_fija_swap = igual_curva_sube_spread.pv_pata_fija_swap - igual_curva_sube_spread.accrued_value_fija_swap
print(f"Ajuste a Valor de Mercado Fija Swap: {ajuste_fija_swap:,.0f}")

Ajuste a Valor de Mercado Fija Swap: -6,306,119,309


In [107]:
ajuste_icpclp_swap = igual_curva_sube_spread.pv_pata_icpclp - igual_curva_sube_spread.accrued_value_icpclp_swap
print(f"Ajuste a Valor de Mercado ICPCLP Swap: {ajuste_icpclp_swap:,.0f}")

Ajuste a Valor de Mercado ICPCLP Swap: 6,290,972,603


In [108]:
print(f"Ajuste a Valor de Mercado Swap: {ajuste_fija_swap + ajuste_icpclp_swap:,.0f}")

Ajuste a Valor de Mercado Swap: -15,146,706
